# scPEFT in-context fine-tuning (draft)

This notebook mirrors the GPU/CPU/Torch setup from `scimilarity_codebase_LoRA.ipynb`.
Update the paths and label key once the test files are provided.

In [1]:
# pip install tqdm if you don't already have it
from pathlib import Path
import os, gc, numpy as np, scipy.sparse as sp, torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import json
import scanpy as sc

# Apple-GPU flags
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"]      = "0"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0"

/Users/kylekimler/miniforge3/envs/scvi-tools/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set up parameters
MODEL_DIR   = Path("/Users/kylekimler/Projects/flagship/scimilarity/data/model_v1.1")
H5AD_PATH   = Path("/Users/kylekimler/Projects/GCA/meta_datasets/20250606_CxG_upload_checkpoint.h5ad")
LABEL_KEY   = "cell_type_atlas"
TEST_SIZE   = 0.20
RAND_SEED   = 42
BATCH = 32  # lower if kernel still crashes 

In [3]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "mps" else torch.float32
print(f"Embedding on {DEVICE} (dtype={DTYPE})…")

# Ensure deterministic splitting
rng = np.random.default_rng(RAND_SEED)
os.environ["PYTHONHASHSEED"] = str(RAND_SEED)

Embedding on mps (dtype=torch.float16)…


In [4]:
from scimilarity.training_models import MetricLearning

def load_encoder(model_dir: Path) -> MetricLearning:
    with open(model_dir / "hyperparameters.json") as fh:
        hparams = json.load(fh)

    # n_genes = length of gene_order.tsv (authoritative)
    with open(model_dir / "gene_order.tsv") as fh:
        gene_order = [line.strip() for line in fh]
    n_genes = len(gene_order)

    model = MetricLearning(
        n_genes=n_genes,
        latent_dim=hparams["latent_dim"],
        hidden_dim=hparams["hidden_dim"],
        dropout=hparams.get("dropout", 0.5),
        input_dropout=hparams.get("input_dropout", 0.4),
    )

    enc_ckpt = torch.load(model_dir / "encoder.ckpt", map_location="cpu")
    enc_state = {k.replace("encoder.", ""): v for k, v in enc_ckpt["state_dict"].items() if k.startswith("encoder.")}
    model.encoder.load_state_dict(enc_state, strict=False)
    model.encoder.to(DEVICE).eval()
    return model, gene_order

encoder, gene_order = load_encoder(MODEL_DIR)

ModuleNotFoundError: No module named 'scimilarity'

In [ ]:
DEVICE, DTYPE = "mps", torch.float32          # fp32 everywhere
enc_base      = encoder.encoder.eval().to(DEVICE)   # freeze decoder away
for p in enc_base.parameters():               # safety
    p.requires_grad_(False)

## Load and align data

In [ ]:
adata = sc.read_h5ad(H5AD_PATH)

# If gene_symbol column present, set it as index so align_dataset works by symbols
if "gene_symbol" in adata.var.columns and adata.var.index.name != "gene_symbol":
    adata.var.set_index("gene_symbol", inplace=True)

if adata.var.index.has_duplicates:
    dup_mask = adata.var.index.duplicated(keep="first")
    print(f"[warn] Dropping {dup_mask.sum()} duplicated gene symbols; keeping first occurrence.")
    adata = adata[:, ~dup_mask].copy()

In [ ]:
from scimilarity.utils import align_dataset

adata = align_dataset(adata, gene_order)  # Align to genes in scimilarity model
adata = adata[:, gene_order]              # drop any extra genes

## Label setup

In [ ]:
# TODO: update LABEL_KEY or add ontology mapping before training
label_col = LABEL_KEY
labels = adata.obs[label_col].astype(str).values

## Train/test split

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(adata.n_obs), test_size=TEST_SIZE, stratify=labels, random_state=RAND_SEED
)

csr_train, y_train = adata.X[idx_train], labels[idx_train]
csr_test,  y_test  = adata.X[idx_test], labels[idx_test]

## scPEFT setup (LoRA simplified)

In [ ]:
# TODO: install/import scPEFT once paths are provided
# Example (update to actual scPEFT API):
# %pip install scpeft
# from scpeft import ScPEFTConfig, attach_peft
# peft_cfg = ScPEFTConfig(...)
# enc_peft = attach_peft(enc_base, peft_cfg)
# enc_peft.train()

## Embeddings and evaluation

In [ ]:
# TODO: embed train/test with enc_peft (or enc_base for baseline) and evaluate
# Example (update to match your embedding pipeline):
# Z_train = ...
# Z_test  = ...
# knn = KNeighborsClassifier(n_neighbors=50, metric="cosine", weights="distance").fit(Z_train, y_train)
# acc = accuracy_score(y_test, knn.predict(Z_test))
# print(f"scPEFT KNN accuracy: {acc:.4f}")